# 01 - Build the development cohort

I start with 2005 Q1 loans that are still active in December 2006 and no worse than one month late. The target is 180+ DPD during 2007. The snapshot decides who qualifies; the model fields come from origination.

Notebook 4 runs the same steps one year later. Both call `build_cohort()` in `cohort.py` so the outcome rule is in one place.


In [1]:
import os
from pathlib import Path

from cohort import build_cohort

LOCAL_DATA = Path(r"E:\CRM Projects\Data")
if "POCKET_CRM_DATA" not in os.environ and LOCAL_DATA.is_dir():
    os.environ["POCKET_CRM_DATA"] = str(LOCAL_DATA)
DATA = Path(os.environ.get("POCKET_CRM_DATA", "./data")).expanduser().resolve()
TARGET = "default_180_plus_2007"


## 2007 outcome

A `06` or higher in any month counts as default. I call a loan a nondefault only when its reporting covers the year or ends with a normal payoff. Unresolved loans stay in the audit file.


In [2]:
# Find the pair of raw files together, also inside extracted subfolders.
source_dirs = [DATA] + [p.parent for p in DATA.rglob("perf_2005Q1.txt")]
source_dirs = list(dict.fromkeys(source_dirs))
source_dirs = [d for d in source_dirs if (d / "perf_2005Q1.txt").is_file()
               and (d / "orig_2005Q1.txt").is_file()]
if not source_dirs:
    raise FileNotFoundError(
        f"Need orig_2005Q1.txt and perf_2005Q1.txt under {DATA}"
    )
if len(source_dirs) > 1:
    raise ValueError(f"More than one 2005 Q1 source folder: {source_dirs}")

audit = build_cohort(source_dirs[0], "2005Q1", "200612", 2007)
model = audit.loc[audit[TARGET].notna()].copy()
model[TARGET] = model[TARGET].astype("int8")

print("Eligible / modeled / defaults / unresolved:", len(audit), len(model),
      model[TARGET].sum(), audit[TARGET].isna().sum())
print("Default rate:", f"{model[TARGET].mean():.3%}")
print("DTI 999:", audit["dti"].eq(999).sum())

audit.to_parquet(DATA / "audit_2005q1_dec2006_2007.parquet", index=False)
model.to_parquet(DATA / "model_2005q1_dec2006_2007.parquet", index=False)


Eligible / modeled / defaults / unresolved: 286317 286188 563 129
Default rate: 0.197%
DTI 999: 8855
